In [ ]:
# --- Xray Cloud: Release Test Report (Unique Latest Run per Test, Charts & Graphs) ---

import requests, getpass, os, json, pandas as pd, datetime
from collections import Counter
from datetime import datetime as dt
from IPython.display import display
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = ["Segoe UI Emoji", "DejaVu Sans"]
import warnings, glob, pathlib

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")
import os, glob, pathlib

user_dir = pathlib.Path.home()
for folder in [pathlib.Path.cwd(), user_dir / "Downloads", user_dir / "Documents", user_dir / "Desktop"]:
    matches = glob.glob(str(folder / "**/2025 Releases Metrics.*"), recursive=True)
if matches:
    print(f"Found: {matches[0]}")

# ============================================================
# Global Config
# ============================================================
XRAY_AUTH_URL    = "https://xray.cloud.getxray.app/api/v2/authenticate"
XRAY_GRAPHQL_URL = "https://xray.cloud.getxray.app/api/v2/graphql"
CACHE_DIR = "xray_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# Disable SSL verify warning (local only)
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
VERIFY_SSL = False

# ============================================================
# Auth
# ============================================================

CID  = getpass.getpass("XRAY_CLIENT_ID: ")
CSEC = getpass.getpass("XRAY_CLIENT_SECRET: ")


def auth():
    r = requests.post(
        XRAY_AUTH_URL,
        json={"client_id": CID, "client_secret": CSEC},
        timeout=30,
        verify=VERIFY_SSL,
    )
    r.raise_for_status()
    return r.text.strip('"')


TOKEN = auth()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

# ============================================================
# Helper function: Graph QL->Xray API endpoint
# ============================================================

def gql(query, variables=None):
    r = requests.post(
        XRAY_GRAPHQL_URL,
        json={"query": query, "variables": variables or {}},
        headers=HEADERS,
        timeout=60,
        verify=VERIFY_SSL,
    )
    r.raise_for_status()
    data = r.json()
    if "errors" in data:
        raise SystemExit(f"GraphQL errors: {data['errors']}")
    return data["data"]

# ============================================================
# Load Releases (Auto-search & validate)
# ============================================================
print("\n Searching for release mapping file (xlsx/csv)...")

file_patterns = [
    "2025 Releases Metrics.*",
    "2024 Releases Metrics.*",
    "releases.*",
]

user_dir = pathlib.Path.home()
search_paths = [
    pathlib.Path.cwd(),
    user_dir / "Downloads",
    user_dir / "Documents",
    user_dir / "Desktop",
]

found_files = []
for base in search_paths:
    for pat in file_patterns:
        matches = glob.glob(str(base / "**" / pat), recursive=True)
        for m in matches:
            # Skip Windows shortcut & URL link files
            if not (m.lower().endswith(".url") or m.lower().endswith(".lnk")):
                found_files.append(m)

if not found_files:
    raise FileNotFoundError(
        "Could not find a valid release mapping file.\n"
        "Looked for '2025 Releases Metrics.*' or 'releases.*' in Desktop/Documents/Downloads/current folder.\n"
        "Please save your file as '2025 Releases Metrics.xlsx' or 'releases.csv'."
    )

# Prefer Excel over CSV if both exist
found_files.sort(key=lambda x: (x.lower().endswith(".csv"), len(x)))
file_path = found_files[0]

print(f"Using release mapping at:\n   {file_path}")

# ============================================================
# Load file safely
# ============================================================
if file_path.lower().endswith(".csv"):
    releases_df_raw = pd.read_csv(file_path)
else:
    try:
        releases_df_raw = pd.read_excel(file_path, sheet_name=0, engine="openpyxl")
    except Exception as e:
        print(f" openpyxl failed ({e}). Retrying with xlrd...")
        releases_df_raw = pd.read_excel(file_path, sheet_name=0, engine="xlrd")

# ============================================================
# Clean & standardize columns
# ============================================================
releases_df_raw.columns = [c.strip() for c in releases_df_raw.columns]

df = (
    releases_df_raw.rename(
        columns={
            c: "Release"
            for c in releases_df_raw.columns
            if c.strip().lower() == "release"
        }
    )
    .rename(
        columns={
            c: "Key"
            for c in releases_df_raw.columns
            if c.strip().lower()
            in ["key", "plan", "test plan", "test plan key", "plan key"]
        }
    )
    .copy()
)

if not {"Release", "Key"}.issubset(df.columns):
    raise ValueError(
        f"Expected columns 'Release' and 'Key' not found. Got columns: {list(releases_df_raw.columns)}"
    )

df["Release"] = df["Release"].astype(str).str.strip()
df["Key"] = df["Key"].astype(str).str.strip()

# ============================================================
# Filter by year (current + last)
# ============================================================

def get_year_from_release(r):
    try:
        prefix = str(r).split(".")[0]
        return 2000 + int(prefix)
    except Exception:
        return None

df["year"] = df["Release"].apply(get_year_from_release)

import datetime as _dt
_current_year = _dt.datetime.now().year
_years_to_include = {_current_year, _current_year - 1}
filtered_releases = df[df["year"].isin(_years_to_include)].dropna(subset=["Key"]).copy()

if filtered_releases.empty:
    raise ValueError(
        f"No releases found for years: {_years_to_include}.\n"
        "Check your 'Release' values (e.g. 25.8R for 2025, 24.4R for 2024)."
    )

print("\n Releases selected (this year + last year):")
display(filtered_releases[["Release", "Key"]].head(20))

# ============================================================
# Queries
# ============================================================
Q_TESTPLANS = """
query($jql:String!, $limit:Int!, $start:Int!) {
  getTestPlans(jql:$jql, limit:$limit, start:$start) {
    results { issueId jira(fields:["key","summary","status"]) }
  }
}
"""
Q_PLAN_EXECS = """
query($planId:String!, $limit:Int!, $start:Int!) {
  getTestPlan(issueId:$planId) {
    testExecutions(limit:$limit, start:$start) {
      total start limit
      results { issueId }
    }
  }
}
"""
Q_PLAN_TESTS = """
query($planId:String!, $limit:Int!, $start:Int!) {
  getTestPlan(issueId:$planId) {
    tests(limit:$limit, start:$start) {
      total start limit
      results { jira(fields:["key"]) }
    }
  }
}
"""
Q_RUNS = """
query($execIds:[String], $limit:Int!, $start:Int!) {
  getTestRuns(testExecIssueIds:$execIds, limit:$limit, start:$start) {
    total start limit
    results {
      status { name }
      test { jira(fields:["key"]) }
      startedOn
      finishedOn
    }
  }
}
"""

# ============================================================
# Helpers
# ============================================================

def get_test_plan_status(plan_key):
    d = gql(Q_TESTPLANS, {"jql": f'key = "{plan_key}"', "limit": 1, "start": 0})
    plans = d["getTestPlans"]["results"]
    if not plans:
        return "Unknown"
    status = plans[0]["jira"].get("status")
    if isinstance(status, dict):
        return status.get("name", "Unknown")
    return str(status).strip().title() if status else "Unknown"


def cache_path_for(plan_key):
    return os.path.join(CACHE_DIR, f"{plan_key}.parquet")


def save_to_cache(plan_key, df):
    if df is None:
        print(f" Skipping cache for {plan_key} (no data).")
        return
    path = cache_path_for(plan_key)
    df.to_parquet(path, index=False)
    print(f"Saved cache for {plan_key} → {path}")


def load_from_cache(plan_key):
    path = cache_path_for(plan_key)
    if os.path.exists(path):
        print(f"Using cached data for {plan_key}")
        return pd.read_parquet(path)
    return None

# ============================================================
# Fetch from Xray
# ============================================================

def fetch_release_data(plan_key):
    MAX_RETRIES = 3  # you can adjust this if needed

    def gql_with_reindex_retry(query, variables, context_label):
        """Call gql(), handling Jira re-index errors with up to MAX_RETRIES."""
        retry_count = 0
        while retry_count < MAX_RETRIES:
            try:
                return gql(query, variables)
            except SystemExit as e:
                msg = str(e)
                if "may need to be re-indexed" in msg:
                    retry_count += 1
                    print(
                        f"\n Jira indexing issue detected! (attempt {retry_count}/{MAX_RETRIES})"
                    )
                    print(f"   Context: {context_label}")
                    print(
                        "   Some issues could not be fetched because the Jira project index is outdated."
                    )
                    print(
                        "   Ask your Jira admin to re-index the project, then retry.\n"
                    )
                    ans = (
                        input(
                            " After re-indexing, press Enter to retry this step (or type 'q' to abort): "
                        )
                        .strip()
                        .lower()
                    )
                    if ans == "q":
                        print(" Aborting current operation per user request.\n")
                        return None
                    continue  # retry again
                else:
                    print("\n GraphQL query failed with an unexpected error:")
                    print(f"   Context: {context_label}")
                    print(f"   {msg}\n")
                    return None
            except Exception as e:
                print("\n Unexpected exception during GraphQL call:")
                print(f"   Context: {context_label}")
                print(f"   {str(e)}\n")
                return None

        print(f" Maximum retries ({MAX_RETRIES}) reached for {context_label}.")
        print("   Please ensure Jira re-indexing is complete and rerun this script.\n")
        return None

    # -------- 1) Get Test Plan --------
    plan_d = gql_with_reindex_retry(
        Q_TESTPLANS,
        {"jql": f'key = "{plan_key}"', "limit": 1, "start": 0},
        "getTestPlans",
    )
    if plan_d is None:
        return None

    if not plan_d.get("getTestPlans", {}).get("results"):
        print(f" No Test Plan found for {plan_key}")
        return None

    plan_id = plan_d["getTestPlans"]["results"][0]["issueId"]
    plan_status = (
        plan_d["getTestPlans"]["results"][0]
        .get("jira", {})
        .get("status", {})
        .get("name", "Unknown")
    )

    # -------- 2) Planned tests (scope) --------
    planned = set()
    start = 0
    while True:
        d = gql_with_reindex_retry(
            Q_PLAN_TESTS,
            {"planId": plan_id, "limit": 200, "start": start},
            "getTestPlan.tests (planned scope)",
        )
        if d is None:
            return None

        block = d.get("getTestPlan", {}).get("tests", {})
        for t in block.get("results", []):
            k = (t.get("jira") or {}).get("key")
            if k:
                planned.add(k)

        start += block.get("limit", 0)
        total = block.get("total", 0)
        if start >= total:
            break

    df_plan = pd.DataFrame(
        [
            {
                "plan_key": plan_key,
                "plan_status": plan_status,
                "source": "PLANNED_SCOPE",
                "test_key": k,
            }
            for k in planned
        ]
    )

    # -------- 3) Gather Test Execution issueIds --------
    exec_ids, all_runs = [], []
    start = 0
    while True:
        d = gql_with_reindex_retry(
            Q_PLAN_EXECS,
            {"planId": plan_id, "limit": 100, "start": start},
            "getTestPlan.testExecutions",
        )
        if d is None:
            return None

        te = d.get("getTestPlan", {}).get("testExecutions", {})
        exec_ids.extend(
            [x.get("issueId") for x in te.get("results", []) if x.get("issueId")]
        )

        start += te.get("limit", 0)
        total = te.get("total", 0)
        if start >= total:
            break

    # -------- 4) Fetch Test Runs in batches --------
    if exec_ids:
        BATCH = 50
        for i in range(0, len(exec_ids), BATCH):
            batch = exec_ids[i : i + BATCH]
            start = 0
            limit = 100
            while True:
                data = gql_with_reindex_retry(
                    Q_RUNS,
                    {"execIds": batch, "limit": limit, "start": start},
                    "getTestRuns",
                )
                if data is None:
                    return None

                runs = data.get("getTestRuns", {})
                for tr in runs.get("results", []):
                    sname = (tr.get("status") or {}).get("name") or "UNKNOWN"
                    test_key = (
                        (tr.get("test") or {})
                        .get("jira", {})
                        .get("key")
                    )
                    all_runs.append(
                        {
                            "plan_key": plan_key,
                            "plan_status": plan_status,
                            "source": "EXECUTION_RUN",
                            "test_key": test_key,
                            "status": sname,
                            "finishedOn": tr.get("finishedOn"),
                        }
                    )

                start += runs.get("limit", 0)
                total = runs.get("total", 0)
                if start >= total:
                    break

    df_exec = pd.DataFrame(all_runs)
    return pd.concat([df_plan, df_exec], ignore_index=True)

# ============================================================
# Dynamic Cache
# ============================================================

def fetch_or_load(plan_key):
    path = cache_path_for(plan_key)
    plan_status = get_test_plan_status(plan_key)
    print(f"Plan {plan_key} status: {plan_status}")
    done_statuses = {"done", "closed", "completed"}

    if plan_status.lower() not in done_statuses:
        print(f"Active → fetching fresh data from Xray...")
        df = fetch_release_data(plan_key)
        save_to_cache(plan_key, df)
        return df
    if os.path.exists(path):
        try:
            cached = pd.read_parquet(path)
            print(f"Using cached data for {plan_key}")
            return cached
        except Exception:
            print("Cache failed → refetching")
    df = fetch_release_data(plan_key)
    if df is None:
        print(f" Failed to fetch data for {plan_key}. (Aborted or retries exhausted)")
        return None  # or return an empty DataFrame with the right schema
    save_to_cache(plan_key, df)
    return df

# ============================================================
# Compute Release Metrics
# ============================================================

def compute_release_metrics(df_all):
    """Compute release metrics ensuring alignment between planned and executed tests."""
    summary = []

    # Derive mapping between plan_key and the release info
    rel_map = (
        df_all.groupby("plan_key")[["Release_Name", "Release_Key"]]
        .first()
        .to_dict("index")
    )

    for rel in df_all["plan_key"].unique():
        subset = df_all[df_all["plan_key"] == rel]
        plan_status = (
            subset["plan_status"].dropna().unique()[0]
            if "plan_status" in subset.columns
            else "Unknown"
        )

        # --- Planned tests list ---
        planned_tests = (
            subset[subset["source"] == "PLANNED_SCOPE"]["test_key"]
            .unique()
            .tolist()
        )
        planned = len(planned_tests)

        # --- Execution runs (latest per test only for planned scope) ---
        exec_subset = subset[subset["source"] == "EXECUTION_RUN"].copy()
        exec_subset = exec_subset[exec_subset["test_key"].isin(planned_tests)]

        if "finishedOn" in exec_subset.columns:
            exec_subset["finishedOn"] = pd.to_datetime(
                exec_subset["finishedOn"], errors="coerce"
            )
            exec_subset = exec_subset.sort_values(
                by=["test_key", "finishedOn"], ascending=True
            )
            latest_runs = exec_subset.groupby("test_key").tail(1)
        else:
            latest_runs = exec_subset.drop_duplicates(
                subset=["test_key"], keep="last"
            )

        # --- Status counts ---
        status_list = ["TO DO", "PASSED", "FAILED", "BLOCKED", "EXECUTING", "DESCOPED"]
        counts = latest_runs["status"].value_counts().reindex(
            status_list, fill_value=0
        )
        (
            todo,
            passed,
            failed,
            blocked,
            executing,
            descoped,
        ) = (
            counts["TO DO"],
            counts["PASSED"],
            counts["FAILED"],
            counts["BLOCKED"],
            counts["EXECUTING"],
            counts["DESCOPED"],
        )

        # --- Actual Execution (raw effort of pass/fail) ---
        actual_execution = len(
            subset[
                (subset["source"] == "EXECUTION_RUN")
                & (subset["status"].isin(["PASSED", "FAILED"]))
            ]
        )

        # --- Add missing To Dos ---
        known_status_tests = set(latest_runs["test_key"])
        missing_tests = set(planned_tests) - known_status_tests
        todo += len(missing_tests)

        # --- Metrics ---
        executed_tests = passed + failed + blocked + executing
        pass_rate = round(
            (passed / (passed + failed) * 100), 2
        ) if (passed + failed) else 0
        exec_coverage = round(
            (executed_tests / planned * 100), 2
        ) if planned else 0
        completion = round(
            ((passed + failed + descoped + blocked + executing) / planned * 100), 2
        ) if planned else 0
        remaining = round(
            ((planned - (passed + failed + descoped + blocked + executing)) / planned * 100),
            2,
        ) if planned else 0

        # --- Map Release/Key for display ---
        release_name = rel_map.get(rel, {}).get("Release_Name", "")
        release_key = rel_map.get(rel, {}).get("Release_Key", "")

        # --- Append to summary ---
        summary.append(
            {
                "Release": release_name,
                "Key": release_key,
                "Status": plan_status,
                "Planned Tests": planned,
                "Actual Execution (Raw)": actual_execution,
                "Execution Coverage %": exec_coverage,
                "Pass %": pass_rate,
                "Completion %": completion,
                "Remaining %": remaining,
                "To Do": todo,
                "Passed": passed,
                "Failed": failed,
                "Blocked": blocked,
                "Executing": executing,
                "Descoped": descoped,
            }
        )

    cols = [
        "Release",
        "Key",
        "Status",
        "Planned Tests",
        "Actual Execution (Raw)",
        "Execution Coverage %",
        "Pass %",
        "Completion %",
        "Remaining %",
        "To Do",
        "Passed",
        "Failed",
        "Blocked",
        "Executing",
        "Descoped",
    ]

    return pd.DataFrame(summary)[cols]

# ============================================================
# Run for Releases (Dynamic)
# ============================================================
all_dfs = []
for _, row in filtered_releases.iterrows():
    rel = row["Release"]
    key = row["Key"]
    print(f"\n Processing {rel} ({key})...")
    df_piece = fetch_or_load(key)
    df_piece["Release_Name"] = rel  # <-- add release name column
    df_piece["Release_Key"] = key   # <-- add key column
    all_dfs.append(df_piece)

df_all = pd.concat(all_dfs, ignore_index=True)

# ============================================================
# Compute and style Release Test Metrics Dashboard
# ============================================================
dashboard_df = compute_release_metrics(df_all)
print("\n Release metrics calculated for", len(dashboard_df), "releases.")

# ============================================================
# Styled Display & Save
# ============================================================

def highlight_health(val, col):
    if pd.isna(val):
        return ""
    if col == "Pass %":
        return (
            "background-color:#C8E6C9;"
            if val >= 95
            else ("background-color:#FFF9C4;" if val >= 90 else "background-color:#FFCDD2;")
        )
    if col == "Completion %":
        return (
            "background-color:#DCEDC8;"
            if val >= 95
            else ("background-color:#FFF9C4;" if val >= 90 else "background-color:#FFCDD2;")
        )
    if col == "Remaining %":
        return (
            "background-color:#C8E6C9;"
            if val <= 5
            else ("background-color:#FFF9C4;" if val <= 10 else "background-color:#FFCDD2;")
        )
    return ""

styled_dashboard = (
    dashboard_df.style
    .format(
        {
            "Execution Coverage %": "{:.2f}%",
            "Pass %": "{:.2f}%",
            "Completion %": "{:.2f}%",
            "Remaining %": "{:.2f}%",
        }
    )
    .apply(
        lambda c: [highlight_health(v, c.name) for v in c]
        if c.name in ["Pass %", "Completion %", "Remaining %"]
        else [""] * len(c)
    )
    .bar(subset=["Execution Coverage %"], color="#2196F3", vmin=0, vmax=100)
    .set_caption("Release Test Metrics Dashboard")
)
display(styled_dashboard)

dashboard_df.to_csv(os.path.join(CACHE_DIR, "release_metrics_dashboard.csv"), index=False)
dashboard_df.to_parquet(
    os.path.join(CACHE_DIR, "release_metrics_dashboard.parquet"), index=False
)
print("\n Dashboard saved in xray_cache/")

# ============================================================
# Visualization Section (Compact Labels + Proper Layout)
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import re

# ---- Global plot style ----
plt.rcParams.update(
    {
        "font.family": ["Segoe UI Emoji", "DejaVu Sans"],
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "figure.figsize": (12, 5),
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    }
)

# ---- Clean up Release labels (remove project-key prefix, _MR, etc) ----

def clean_release_label(name):
    if pd.isna(name):
        return ""
    name = str(name)
    name = re.sub(r"^PROJ[-_]*", "", name)  # Remove project-key prefix
    name = re.sub(r"[Rr]?[_-]*", "", name)  # Remove stray R or underscores
    name = name.replace("MR", " MR")  # Add space before MR
    return name.strip()

dashboard_df["ReleaseLabel"] = dashboard_df["Release"].apply(clean_release_label)

# ------------------------------------------------------------
#  Pass Rate Trend per Release
# ------------------------------------------------------------
plt.figure(figsize=(12, 5))
plt.plot(
    dashboard_df["ReleaseLabel"],
    dashboard_df["Pass %"],
    marker="o",
    color="orange",
    label="Pass Rate",
    linewidth=2,
)
plt.axhline(y=90, color="orange", linestyle="dotted", label="Target: 90%")

for i, v in enumerate(dashboard_df["Pass %"]):
    plt.text(i, v + 0.8, f"{v:.0f}%", ha="center", fontsize=9, fontweight="bold")

plt.title("Pass Rate Trend per Release")
plt.xlabel("Release")
plt.ylabel("Pass Rate (%)")
plt.ylim(0, 105)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.xticks(rotation=30, ha="right")
plt.subplots_adjust(bottom=0.18, top=0.9)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Planned vs Actual Tests per Release
# ------------------------------------------------------------
dashboard_df["Actual Executed"] = (
    dashboard_df["Planned Tests"] * (dashboard_df["Completion %"] / 100)
).round(0)

plt.figure(figsize=(12, 5))
plt.plot(
    dashboard_df["ReleaseLabel"],
    dashboard_df["Planned Tests"],
    marker="o",
    color="orange",
    label="Planned Tests",
    linewidth=2,
)
plt.plot(
    dashboard_df["ReleaseLabel"],
    dashboard_df["Actual Executed"],
    marker="o",
    color="skyblue",
    label="Actual Executed",
    linewidth=2,
)

for i in range(len(dashboard_df)):
    diff = dashboard_df["Actual Executed"][i] - dashboard_df["Planned Tests"][i]
    plt.text(
        i,
        dashboard_df["Actual Executed"][i] + 2,
        f"{diff:+.0f}",
        ha="center",
        fontsize=9,
        color="gray",
    )

plt.title("Release Test Cycle: Planned vs Actual Tests")
plt.xlabel("Release")
plt.ylabel("Number of Tests")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.xticks(rotation=30, ha="right")
plt.subplots_adjust(bottom=0.18, top=0.9)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Test Distribution per Release (6 Statuses)
# ------------------------------------------------------------
status_cols = ["Passed", "Failed", "Blocked", "Executing", "Descoped", "To Do"]
colors = {
    "Passed": "#4CAF50",
    "Failed": "#F44336",
    "Blocked": "#9E9E9E",
    "Executing": "#03A9F4",
    "Descoped": "#795548",
    "To Do": "#FFEB3B",
}

plt.figure(figsize=(12, 6))
bottom_vals = np.zeros(len(dashboard_df))

for status in status_cols:
    plt.bar(
        dashboard_df["ReleaseLabel"],
        dashboard_df[status],
        bottom=bottom_vals,
        label=status,
        color=colors[status],
    )
    for i, val in enumerate(dashboard_df[status]):
        total = dashboard_df.loc[i, "Planned Tests"]
        if val > 0:
            pct = (val / total) * 100
            plt.text(
                i,
                bottom_vals[i] + val / 2,
                f"{pct:.0f}%",
                ha="center",
                va="center",
                fontsize=9,
                color="white",
            )
    bottom_vals += dashboard_df[status]

# Annotate total tests on top
for i, total in enumerate(dashboard_df["Planned Tests"]):
    plt.text(
        i,
        total + 5,
        f"Total: {int(total)}",
        ha="center",
        fontsize=9,
        color="gray",
    )

plt.title("Test Distribution per Release (6 Statuses)")
plt.xlabel("Release")
plt.ylabel("Number of Tests")
plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
plt.xticks(rotation=30, ha="right")
plt.subplots_adjust(bottom=0.18, right=0.85, top=0.9)
plt.tight_layout()
plt.show()
